<a href="https://colab.research.google.com/github/Leonardozepeda04/edt-dataa-pipeline/blob/main/notebooks/aseguradoras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
#URL csv Aseguradoras
url_aseguradoras = "https://raw.githubusercontent.com/Leonardozepeda04/edt-dataa-pipeline/refs/heads/main/data/raw/aseguradoras.csv"


In [3]:
aseguradoras = pd.read_csv(url_aseguradoras)

In [4]:
aseguradoras.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_aseguradora  15 non-null     int64 
 1   nombre          15 non-null     object
 2   pais            13 non-null     object
 3   rating_riesgo   12 non-null     object
dtypes: int64(1), object(3)
memory usage: 612.0+ bytes


In [5]:
#Limpieza de Datos
def limpiar_dataframe(df):

    df.columns = df.columns.str.strip().str.lower()

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

    df = df.replace(r'^\s*$', pd.NA, regex=True)

    df = df.drop_duplicates()

    return df

In [6]:
#Transformaciones Importantes

# 1. Normalizar nombres de países (ejemplo: "ElSalvador" → "El Salvador")
aseguradoras['pais'] = aseguradoras['pais'].astype(str).str.strip()
aseguradoras['pais'] = aseguradoras['pais'].str.replace("ElSalvador", "El Salvador")

# 2. Rellenar valores faltantes con NaN para consistencia
aseguradoras['pais'] = aseguradoras['pais'].replace(["", "nan", "None"], pd.NA)
aseguradoras['rating_riesgo'] = aseguradoras['rating_riesgo'].replace(["", "nan", "None"], pd.NA)

# 3. Normalizar la columna de riesgo (ejemplo: convertir letras a categorías estándar)
map_riesgo = {
    "Alto": "Alto",
    "Medio": "Medio",
    "Bajo": "Bajo",
    "B": "Bajo",
    "D": "Desconocido"  # ejemplo de mapeo
}
aseguradoras['rating_riesgo'] = aseguradoras['rating_riesgo'].map(map_riesgo)

# 4. Convertir a categoría para optimizar memoria
aseguradoras['pais'] = aseguradoras['pais'].astype("category")
aseguradoras['rating_riesgo'] = aseguradoras['rating_riesgo'].astype("category")


In [7]:
#Verificar resultados
print(aseguradoras.head(15))

    id_aseguradora          nombre         pais rating_riesgo
0                1   Aseguradora 1   Costa Rica          Alto
1                2   Aseguradora 2  El Salvador          Bajo
2                3   Aseguradora 3  El Salvador           NaN
3                4   Aseguradora 4   Costa Rica         Medio
4                5   Aseguradora 5  El Salvador          Bajo
5                6   Aseguradora 6          NaN         Medio
6                7   Aseguradora 7    Guatemala          Alto
7                8   Aseguradora 8       Panamá          Bajo
8                9   Aseguradora 9          NaN          Bajo
9               10  Aseguradora 10       Panamá           NaN
10              11  Aseguradora 11     Honduras   Desconocido
11              12  Aseguradora 12  El Salvador          Bajo
12              13  Aseguradora 13     Honduras          Alto
13              14  Aseguradora 14  El Salvador           NaN
14              15  Aseguradora 15  El Salvador          Alto


In [8]:
#Separar datos validos y rechazados
validos = aseguradoras[
    aseguradoras['nombre'].notna() &
    aseguradoras['pais'].notna() &
    aseguradoras['rating_riesgo'].notna()
].copy()

rechazados = aseguradoras[
    aseguradoras['nombre'].isna() |
    aseguradoras['pais'].isna() |
    aseguradoras['rating_riesgo'].isna()
].copy()



In [11]:
# Función para identificar motivos de rechazo
def motivo(row):
    motivos = []
    if pd.isna(row['nombre']):
        motivos.append("nombre_vacio")
    if pd.isna(row['pais']):
        motivos.append("pais_vacio")
    if pd.isna(row['rating_riesgo']):
        motivos.append("riesgo_vacio")
    return ",".join(motivos)
    rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)


In [12]:
#Exportar Archivo curated
aseguradoras.to_csv("aseguradoras_curated.csv", index=False)